# Visualize the knowledge-base collections

Loads every person collection built by `knowledge_base_constructor`, projects the embeddings to 3D with PCA, and plots them interactively. Hover a point to see the person, source type, and the chunk of text it came from.

In [11]:
import sys
from pathlib import Path


def find_project_root(marker="pyproject.toml"):
    path = Path.cwd()
    for candidate in [path, *path.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} in any parent of {path}")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [12]:
import chromadb

from config import VECTORS_DIR

# collections that aren't a person's knowledge base (created by other parts of the project)
EXCLUDED_COLLECTIONS = {"langchain"}

client = chromadb.PersistentClient(path=str(VECTORS_DIR))
person_collections = [c for c in client.list_collections() if c.name not in EXCLUDED_COLLECTIONS]

print("Person collections found:", [c.name for c in person_collections])

Person collections found: ['giorgia-meloni', 'ilaria-salis', 'roberto-vannacci', 'matteo-renzi']


In [13]:
rows = []
for collection in person_collections:
    result = collection.get(include=["embeddings", "documents", "metadatas"])
    for embedding, document, metadata in zip(
        result["embeddings"], result["documents"], result["metadatas"]
    ):
        rows.append(
            {
                "embedding": embedding,
                "chunk": document,
                "name": metadata.get("name", "?"),
                "surname": metadata.get("surname", "?"),
                "source_type": metadata.get("source_type", "?"),
                "person_id": metadata.get("person_id", collection.name),
            }
        )

print(f"{len(rows)} chunks across {len(person_collections)} people")

122 chunks across 4 people


In [14]:
import pandas as pd
from sklearn.decomposition import PCA

df = pd.DataFrame(rows)
coords = PCA(n_components=3).fit_transform(list(df["embedding"]))
df[["x", "y", "z"]] = coords

df["person"] = df["name"] + " " + df["surname"]


def preview(text, max_chars=300):
    text = text[:max_chars] + ("…" if len(text) > max_chars else "")
    # wrap so the hover box doesn't become one giant unreadable line
    words = text.split()
    lines, line = [], ""
    for word in words:
        if len(line) + len(word) > 60:
            lines.append(line)
            line = ""
        line += word + " "
    lines.append(line)
    return "<br>".join(lines)


df["chunk_preview"] = df["chunk"].apply(preview)

In [15]:
import plotly.express as px

fig = px.scatter_3d(
    df,
    x="x",
    y="y",
    z="z",
    color="person",
    symbol="source_type",
    hover_name="person",
    custom_data=["chunk_preview", "source_type"],
    title="Knowledge base chunks, projected to 3D (PCA)",
)
fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>source: %{customdata[1]}<br><br>%{customdata[0]}<extra></extra>"
)
fig.update_layout(height=800)
fig.show()